# 12 Audit

**Stage 4.** Before any number goes into the report, check that the results are right. Each check prints PASS or FAIL. The notebook ends with a table of all checks and writes `results/audit.md` from it, so the file regenerates every time the notebook runs.

Checks 1 and 2 test the data split. Check 1 also tests that every scores file holds exactly the holdout users, with treatment, visit and conversion equal to the data file. Check 3 tests that the end of every Qini curve equals the observed average effect, which it reads from a file made independently of the scores. Check 4 tests that the intervals contain the estimates. Check 5 tests that no model beats the known answer on planted data. Checks 6 and 7 test whether the H1 result is stable, and check 6 also reruns the visit tests for 12 more bootstrap seeds. Check 8 tests that a scores file can be reproduced by a refit. Check 9 is one extra check that the 5M scores file holds exactly the holdout users, in order.

Checks 6 and 7 fail. The notebook explains each failure, names the report claims it affects, and ends with a list of claims the report must not make.

This notebook reads results and refits only what a check needs. It changes no existing result file. The spec calls it `10_audit`. Notebooks 10 and 11 are already used on another branch, so this one is numbered 12.

In [1]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
RESULTS = ROOT / "results"

from uplift.data import FEATURES, TREATMENT, OUTCOMES, TRAIN_SIZES, HOLDOUT_SIZE, make_splits, take
from uplift.learners import SLearner, TLearner, XLearner
from uplift import metrics as m

MODELS = ["response", "s", "t", "x"]
NAMES = {"response": "Response baseline", "s": "S-learner", "t": "T-learner", "x": "X-learner"}
LABELS = {100_000: "100k", 500_000: "500k", 1_000_000: "1m", 5_000_000: "5m"}
N_BOOT = 200           # same as notebook 04
AUDIT = []             # one dict per check, filled by record()
SCORES = {}            # scores files by training size, filled in check 1

def record(n, name, passed, comment):
    assert "\n" not in comment and "|" not in comment
    AUDIT.append(dict(check=f"{n}. {name}", result="PASS" if passed else "FAIL", comment=comment))
    print(f"Check {n}, {name}: {'PASS' if passed else 'FAIL'}")
    print("  " + comment)

def md_table(d):
    """A small markdown table from a data frame of strings."""
    head = "| " + " | ".join(d.columns) + " |"
    sep = "| " + " | ".join("---" for _ in d.columns) + " |"
    body = ["| " + " | ".join(str(v) for v in r) + " |" for r in d.itertuples(index=False)]
    return "\n".join([head, sep] + body)

df = pd.read_parquet(ROOT / "data" / "criteo.parquet")
test_ids, pool_ids = make_splits(len(df))
kept = json.load(open(RESULTS / "kept_features.json"))
best = json.load(open(RESULTS / "best_params.json"))
print(f"full file {len(df):,} rows | holdout {len(test_ids):,} | training pool {len(pool_ids):,}")

full file 13,979,592 rows | holdout 2,000,000 | training pool 11,979,592


## Check 1. No leakage

For every training size, the training ids `pool_ids[:n]` must share no id with `test_ids`. We also confirm that the holdout has the expected size and no repeated id.

Then we test each scores file (100k, 500k, 1m, 5m). Its index must be exactly the 2,000,000 holdout ids from `make_splits`, in that order, and its treatment, visit and conversion columns must equal the data file at those ids. A file that cannot be read or does not match is recorded as a FAIL.

What this check does not prove: that a model was trained on `pool_ids[:n]`. A scores file only holds scores for the holdout. Check 8 is the one that proves it, by refitting at 100k on `pool_ids[:100_000]` and matching the stored scores bit for bit.

In [2]:
overlap = {n: len(np.intersect1d(pool_ids[:n], test_ids)) for n in TRAIN_SIZES}
for n, k in overlap.items():
    print(f"train {n:>9,}: {k} ids shared with the holdout")
holdout_ok = len(test_ids) == HOLDOUT_SIZE and len(np.unique(test_ids)) == len(test_ids)

# Every scores file must be built on exactly the holdout rows. This tests the files, not the training:
# check 8 is the one that proves a model was trained on pool_ids[:n] (bit exact refit at 100k).
file_rows = []
for n in TRAIN_SIZES:
    name = f"scores_{LABELS[n]}.parquet"
    try:
        sc = pd.read_parquet(RESULTS / name)
        SCORES[n] = sc
        ids = sc.index.to_numpy()
        ids_ok = len(ids) == len(test_ids) and bool(np.array_equal(ids, test_ids))
        cols_ok = ids_ok and all(np.array_equal(sc[c].to_numpy(), df[c].to_numpy()[ids]) for c in [TREATMENT] + list(OUTCOMES))
        file_rows.append(dict(file=name, rows=len(sc), ids_are_holdout=ids_ok, columns_match_data=bool(cols_ok), problem=""))
    except Exception as e:
        file_rows.append(dict(file=name, rows=0, ids_are_holdout=False, columns_match_data=False, problem=type(e).__name__))
files_check = pd.DataFrame(file_rows)
display(files_check)
n_files_ok = int((files_check["ids_are_holdout"] & files_check["columns_match_data"]).sum())
files_ok = n_files_ok == len(files_check)
record(1, "No leakage", all(k == 0 for k in overlap.values()) and holdout_ok and files_ok,
       f"The holdout has {len(test_ids):,} unique ids and shares {sum(overlap.values())} ids with the training samples of {', '.join(LABELS[n] for n in TRAIN_SIZES)} rows. "
       f"{n_files_ok} of {len(files_check)} scores files have exactly the holdout ids as index, in make_splits order, with treatment, visit and conversion equal to the data file. "
       f"This does not show which rows a model was trained on, check 8 does that at 100k by a bit exact refit on pool_ids[:n].")

train   100,000: 0 ids shared with the holdout
train   500,000: 0 ids shared with the holdout
train 1,000,000: 0 ids shared with the holdout
train 5,000,000: 0 ids shared with the holdout


,file,rows,ids_are_holdout,columns_match_data,problem
0,scores_100k.parquet,2000000,True,True,
1,scores_500k.parquet,2000000,True,True,
2,scores_1m.parquet,2000000,True,True,
3,scores_5m.parquet,2000000,True,True,


Check 1, No leakage: PASS
  The holdout has 2,000,000 unique ids and shares 0 ids with the training samples of 100k, 500k, 1m, 5m rows. 4 of 4 scores files have exactly the holdout ids as index, in make_splits order, with treatment, visit and conversion equal to the data file. This does not show which rows a model was trained on, check 8 does that at 100k by a bit exact refit on pool_ids[:n].


## Check 2. Treated share

The share of treated users in the holdout and in each training sample must be within 1 percentage point of 85 percent. The file is ordered, so a bad sample could be almost all treated or all control.

In [3]:
treat = df[TREATMENT].to_numpy()
shares = {"holdout": treat[test_ids].mean()}
shares.update({f"train {LABELS[n]}": treat[pool_ids[:n]].mean() for n in TRAIN_SIZES})
for k, v in shares.items():
    print(f"{k:>12}: treated share {v:.4f}")
worst = max(abs(v - 0.85) for v in shares.values())
record(2, "Treated share", worst <= 0.01,
       f"Treated share runs from {min(shares.values()):.4f} to {max(shares.values()):.4f} across the holdout and the 4 training samples. The largest gap from 0.85 is {worst * 100:.3f} percentage points.")

     holdout: treated share 0.8497
  train 100k: treated share 0.8505
  train 500k: treated share 0.8506
    train 1m: treated share 0.8503
    train 5m: treated share 0.8502
Check 2, Treated share: PASS
  Treated share runs from 0.8497 to 0.8506 across the holdout and the 4 training samples. The largest gap from 0.85 is 0.061 percentage points.


## Check 3. The end of each Qini curve equals the average effect

The last point of `qini_curve` is the effect over everyone targeted, so it must equal the observed treated rate minus the control rate, to within 1e-6.

This is an identity. The last point of the curve equals the average effect of the very rows and outcomes it was computed from, whatever the scores are, so a model with random scores passes it. On its own the test would prove little. What makes it a real check is the source of the average effect. We do not compute it from the scores file. We take it from `results/ate_visit_conversion.csv`, the row with `sample == 'hold'`, which was written by a different notebook straight from the data file. A scores file with wrong rows or wrong outcome columns then fails. We test every model and outcome in all four scores files.

In [4]:
# Identity plus an independent source check. The curve end equals the average effect of the same rows whatever the
# scores are (a random score passes the identity), so the reference is read from results/ate_visit_conversion.csv
# (sample == 'hold', column ate), not from the scores file being tested.
ate_file = pd.read_csv(RESULTS / "ate_visit_conversion.csv")
ate_hold = ate_file[ate_file["sample"] == "hold"].set_index("outcome")["ate"]
print("observed average effect on the holdout from ate_visit_conversion.csv:", ate_hold.round(6).to_dict())
rows = []
for n in TRAIN_SIZES:
    if n not in SCORES:
        print(f"{LABELS[n]}: scores file missing, already a FAIL in check 1")
        continue
    sc = SCORES[n]
    t_ = sc[TREATMENT].to_numpy()
    for outcome in OUTCOMES:
        y_ = sc[outcome].to_numpy()
        ate_own = y_[t_ == 1].mean() - y_[t_ == 0].mean()
        for mod in MODELS:
            x_, q_ = m.qini_curve(y_, t_, sc[f"{mod}_{outcome}"])
            rows.append(dict(file=LABELS[n], outcome=outcome, model=mod, last_x=x_[-1], last_q=q_[-1],
                             ate_file=ate_hold[outcome], gap=abs(q_[-1] - ate_hold[outcome]), gap_own=abs(q_[-1] - ate_own)))
curve_end = pd.DataFrame(rows)
expected_curves = len(TRAIN_SIZES) * len(OUTCOMES) * len(MODELS)
print(curve_end.groupby(["file", "outcome"])["gap"].max().map("{:.2e}".format).unstack())
check3_ok = len(curve_end) == expected_curves and bool((curve_end["gap"] < 1e-6).all() and (curve_end["last_x"] == 1.0).all())
record(3, "Curve end equals the average effect", check3_ok,
       f"The curve end matches the observed holdout average effect from ate_visit_conversion.csv for {len(curve_end)} of {expected_curves} curves (4 files, 2 outcomes, 4 models). "
       f"The largest gap is {curve_end['gap'].max() if len(curve_end) else float('nan'):.2e}. The curve end is an identity for its own rows, the independent part is the reference file.")

observed average effect on the holdout from ate_visit_conversion.csv: {'visit': 0.009952, 'conversion': 0.001174}


outcome conversion     visit
file                        
100k      0.00e+00  0.00e+00
1m        0.00e+00  0.00e+00
500k      0.00e+00  0.00e+00
5m        0.00e+00  0.00e+00
Check 3, Curve end equals the average effect: PASS
  The curve end matches the observed holdout average effect from ate_visit_conversion.csv for 32 of 32 curves (4 files, 2 outcomes, 4 models). The largest gap is 0.00e+00. The curve end is an identity for its own rows, the independent part is the reference file.


## Check 4. Intervals contain the estimates

For every row of `qini_results_visit_5m.csv` and `qini_results_conversion_5m.csv`, `ci_lo <= qini_coef <= ci_hi`.

In [5]:
rows = []
for outcome in OUTCOMES:
    r = pd.read_csv(RESULTS / f"qini_results_{outcome}_5m.csv", index_col=0)
    for mod, row in r.iterrows():
        rows.append(dict(outcome=outcome, model=mod, ci_lo=row.ci_lo, qini_coef=row.qini_coef, ci_hi=row.ci_hi,
                         inside=bool(row.ci_lo <= row.qini_coef <= row.ci_hi),
                         margin=min(row.qini_coef - row.ci_lo, row.ci_hi - row.qini_coef)))
inside = pd.DataFrame(rows)
display(inside.round(6))
record(4, "Intervals contain the estimates", bool(inside["inside"].all()),
       f"{int(inside['inside'].sum())} of {len(inside)} rows have ci_lo <= qini_coef <= ci_hi. The smallest margin to an interval end is {inside['margin'].min():.6f}.")

,outcome,model,ci_lo,qini_coef,ci_hi,inside,margin
0,visit,response,0.003155,0.003402,0.003728,True,0.000247
1,visit,s,0.003306,0.003559,0.003837,True,0.000253
2,visit,t,0.003333,0.003579,0.003885,True,0.000246
3,visit,x,0.003434,0.003687,0.004015,True,0.000253
4,conversion,response,0.000418,0.000488,0.000568,True,0.000070
5,conversion,s,0.000407,0.000479,0.000564,True,0.000072
6,conversion,t,0.000261,0.000356,0.000439,True,0.000083
7,conversion,x,0.000361,0.000445,0.000527,True,0.000082


Check 4, Intervals contain the estimates: PASS
  8 of 8 rows have ci_lo <= qini_coef <= ci_hi. The smallest margin to an interval end is 0.000070.


## Check 5. No model beats the oracle on planted data

The oracle score is the true effect, so no learner should rank better than it. We compare only inside planted tests, never against the real data.

**Part A, the committed planted test.** `results/sanity_planted_effect.csv` was made in notebook 04 with a generator seeded by 7. We rebuild the same planted outcome, confirm that the oracle, random and reversed rows of the file are reproduced, and then use the bootstrap noise of the oracle coefficient as the margin. A row passes if it is not above the oracle by more than that margin.

**Part B, the unit test data.** The planted data from `tests/conftest.py` (effect 0.10 for users with `x0 > 0`, zero otherwise). We fit the S, T and X learners on the first half, score the second half, and test each learner minus the oracle with the paired bootstrap. A learner fails if the whole 95 percent interval of its difference to the oracle is above zero.

In [6]:
# Part A. Rebuild the planted test of notebook 04 (same seed, same order of random draws).
rng = np.random.default_rng(7)
def planted_outcome(ids):
    d = take(df, ids)
    tau = 0.05 * (d["f0"] > d["f0"].median()).to_numpy()
    fake = (rng.random(len(d)) < 0.05 + tau * d[TREATMENT].to_numpy()).astype(int)
    return d[TREATMENT].to_numpy(), fake, tau
_ = planted_outcome(pool_ids[:500_000])              # uses the same draws as the training slice in notebook 04
t_a, y_a, tau_a = planted_outcome(test_ids[:500_000])
rebuilt = {"oracle (true effect)": tau_a, "random": rng.random(len(y_a)), "reversed oracle": -tau_a}
file_a = pd.read_csv(RESULTS / "sanity_planted_effect.csv", index_col=0)["qini_coef"]
rebuilt_coef = pd.Series({k: m.qini_coefficient(y_a, t_a, v) for k, v in rebuilt.items()})
rep_gap = (rebuilt_coef - file_a[rebuilt_coef.index]).abs()
print("rebuilt versus committed (oracle, random, reversed):\n", pd.DataFrame({"rebuilt": rebuilt_coef, "committed": file_a[rebuilt_coef.index], "gap": rep_gap}).round(6))
boot_a = m.bootstrap_metrics(y_a, t_a, {"oracle": tau_a}, n_boot=N_BOOT)
ci_a = m.bootstrap_ci(boot_a).loc["oracle"]
noise_a = float((ci_a["hi"] - ci_a["lo"]) / 2)
oracle_a = float(file_a["oracle (true effect)"])
part_a = pd.DataFrame({"qini_coef": file_a, "oracle_plus_noise": oracle_a + noise_a})
part_a["ok"] = part_a["qini_coef"] <= part_a["oracle_plus_noise"]
display(part_a.round(6))
reproduced = bool((rep_gap < 1e-6).all())
part_a_ok = bool(part_a["ok"].all())
learner_a = float(file_a["T-learner"])

rebuilt versus committed (oracle, random, reversed):
                        rebuilt  committed  gap
oracle (true effect)  0.006317   0.006317  0.0
random               -0.000133  -0.000133  0.0
reversed oracle      -0.006017  -0.006017  0.0


,qini_coef,oracle_plus_noise,ok
oracle (true effect),0.006317,0.006823,True
T-learner,0.005956,0.006823,True
random,-0.000133,0.006823,True
reversed oracle,-0.006017,0.006823,True


In [7]:
# Part B. The planted data of tests/conftest.py, built with the same code as the test fixture.
rb = np.random.default_rng(0)
nb = 200_000
XB = pd.DataFrame(rb.normal(size=(nb, 4)), columns=["x0", "x1", "x2", "x3"]).astype("float32")
tb = (rb.random(nb) < 0.85).astype(int)
tau_b = 0.10 * (XB["x0"] > 0).to_numpy()
yb = (rb.random(nb) < 0.05 + tau_b * tb).astype(int)

FAST = dict(n_estimators=60, learning_rate=0.1, num_leaves=15, min_child_samples=200)   # as in tests/test_learners.py
half = nb // 2
yh, th = yb[half:], tb[half:]
sc_b = {"oracle": tau_b[half:]}
for name, cls in (("s", SLearner), ("t", TLearner), ("x", XLearner)):
    sc_b[name] = cls(FAST).fit(XB.iloc[:half], tb[:half], yb[:half]).predict_uplift(XB.iloc[half:])
boot_b = m.bootstrap_metrics(yh, th, sc_b, n_boot=N_BOOT)
point_b = {k: m.qini_coefficient(yh, th, v) for k, v in sc_b.items()}
rows = []
for name in ("s", "t", "x"):
    d = m.paired_difference(boot_b, name, "oracle", point_b[name], point_b["oracle"])
    rows.append(dict(learner=NAMES[name], qini_coef=point_b[name], oracle=point_b["oracle"], diff_vs_oracle=d["diff"],
                     ci_lo=d["lo"], ci_hi=d["hi"], above_oracle=bool(d["lo"] > 0)))
part_b = pd.DataFrame(rows)
display(part_b.round(6))
part_b_ok = not part_b["above_oracle"].any()

record(5, "Bounded by the oracle", reproduced and part_a_ok and part_b_ok,
       f"Committed planted test: the oracle gives {oracle_a:.6f}, the T-learner {learner_a:.6f}, noise margin {noise_a:.6f}, and the oracle, random and reversed rows are reproduced to {rep_gap.max():.1e}. "
       f"Unit test data: the largest learner minus oracle is {part_b['diff_vs_oracle'].max():.6f} and {int(part_b['above_oracle'].sum())} of 3 learners have an interval entirely above zero.")

,learner,qini_coef,oracle,diff_vs_oracle,ci_lo,ci_hi,above_oracle
0,S-learner,0.012274,0.012451,-0.000178,-0.000660,0.000874,False
1,T-learner,0.012185,0.012451,-0.000266,-0.000846,0.000806,False
2,X-learner,0.012474,0.012451,0.000023,-0.000652,0.001030,False


Check 5, Bounded by the oracle: PASS
  Committed planted test: the oracle gives 0.006317, the T-learner 0.005956, noise margin 0.000506, and the oracle, random and reversed rows are reproduced to 4.8e-07. Unit test data: the largest learner minus oracle is 0.000023 and 0 of 3 learners have an interval entirely above zero.


## Check 6. Bootstrap stability of the H1 test

We repeat the H1 test of notebook 04 exactly: the same scores, the same paired bootstrap of all four models together with 200 resamples, the same `paired_difference` against the response baseline, and the Bonferroni correction over 3 learners (p times 3, at most 1). Only the bootstrap seed changes, from 42 to 7. The verdict rule is the one printed by notebook 04: a learner is supported when its interval is above zero and its Bonferroni p value is under 0.05. A verdict that flips means the result is fragile.

In [8]:
s5 = pd.read_parquet(RESULTS / "scores_5m.parquet")
y5 = {o: s5[o].to_numpy() for o in OUTCOMES}; t5 = s5[TREATMENT].to_numpy()
committed = pd.read_csv(RESULTS / "h1_tests_5m.csv")
SEED = 7
rows = []
for outcome in OUTCOMES:
    sc = {mod: s5[f"{mod}_{outcome}"].to_numpy() for mod in NAMES}
    boot = m.bootstrap_metrics(y5[outcome], t5, sc, n_boot=N_BOOT, seed=SEED)
    point = {mod: m.qini_coefficient(y5[outcome], t5, v) for mod, v in sc.items()}
    for mod in ("s", "t", "x"):
        r = m.paired_difference(boot, mod, "response", point[mod], point["response"])
        rows.append(dict(outcome=outcome, model=NAMES[mod], diff_vs_response=r["diff"], ci_lo=r["lo"], ci_hi=r["hi"],
                         p_one_sided=r["p_one_sided"], p_bonferroni=min(1.0, r["p_one_sided"] * 3)))
rerun = pd.DataFrame(rows)
cmp6 = committed.merge(rerun, on=["outcome", "model"], suffixes=("_seed42", "_seed7"))
supported = lambda lo, p: (lo > 0) & (p < 0.05)
cmp6["supported_seed42"] = supported(cmp6["ci_lo_seed42"], cmp6["p_bonferroni_seed42"])
cmp6["supported_seed7"] = supported(cmp6["ci_lo_seed7"], cmp6["p_bonferroni_seed7"])
cmp6["flipped"] = cmp6["supported_seed42"] != cmp6["supported_seed7"]
# p = (k + 1) / (N_BOOT + 1), where k is the number of resamples in which the learner did not beat the response baseline
cmp6["k_seed42"] = (cmp6["p_one_sided_seed42"] * (N_BOOT + 1)).round().astype(int) - 1
cmp6["k_seed7"] = (cmp6["p_one_sided_seed7"] * (N_BOOT + 1)).round().astype(int) - 1
display(cmp6[["outcome", "model", "diff_vs_response_seed42", "diff_vs_response_seed7", "ci_lo_seed42", "ci_lo_seed7",
              "k_seed42", "k_seed7", "p_bonferroni_seed42", "p_bonferroni_seed7", "supported_seed42", "supported_seed7", "flipped"]].round(5))
flips = cmp6[cmp6["flipped"]]
for _, r in flips.iterrows():
    print(f"FRAGILE: {r.outcome} {r.model} goes from supported={r.supported_seed42} to supported={r.supported_seed7}")
flip_text = "no verdict changes" if flips.empty else "verdict flips for " + ", ".join(f"{r.model} on {r.outcome}" for _, r in flips.iterrows()) + " (fragile)"
xv = cmp6[(cmp6.outcome == "visit") & (cmp6.model == "X-learner")].iloc[0]
record(6, "Bootstrap stability", flips.empty,
       f"Seed 7 against seed 42 on 6 learner and outcome tests: {flip_text}. "
       f"Bonferroni p values for visits S, T, X: {', '.join(f'{p:.4f}' for p in cmp6[cmp6.outcome == 'visit'].p_bonferroni_seed7)} (seed 7) against {', '.join(f'{p:.4f}' for p in cmp6[cmp6.outcome == 'visit'].p_bonferroni_seed42)} (seed 42). "
       f"For the X-learner on visits, {int(xv.k_seed42)} of {N_BOOT} resamples did not beat the response baseline at seed 42 and {int(xv.k_seed7)} at seed 7. Explained below.")

,outcome,model,diff_vs_response_seed42,diff_vs_response_seed7,ci_lo_seed42,ci_lo_seed7,k_seed42,k_seed7,p_bonferroni_seed42,p_bonferroni_seed7,supported_seed42,supported_seed7,flipped
0,visit,S-learner,0.00016,0.00016,0.00009,0.00007,0,0,0.01492,0.01493,True,True,False
1,visit,T-learner,0.00018,0.00018,-0.00009,-0.00007,21,15,0.32836,0.23881,False,False,False
2,visit,X-learner,0.00028,0.00029,0.00002,0.00004,4,2,0.07463,0.04478,False,True,True
3,conversion,S-learner,-0.00001,-0.00001,-0.00002,-0.00002,160,165,1.00000,1.00000,False,False,False
4,conversion,T-learner,-0.00013,-0.00013,-0.00021,-0.00020,200,199,1.00000,1.00000,False,False,False
5,conversion,X-learner,-0.00004,-0.00004,-0.00009,-0.00008,188,188,1.00000,1.00000,False,False,False


FRAGILE: visit X-learner goes from supported=False to supported=True
Check 6, Bootstrap stability: FAIL
  Seed 7 against seed 42 on 6 learner and outcome tests: verdict flips for X-learner on visit (fragile). Bonferroni p values for visits S, T, X: 0.0149, 0.2388, 0.0448 (seed 7) against 0.0149, 0.3284, 0.0746 (seed 42). For the X-learner on visits, 4 of 200 resamples did not beat the response baseline at seed 42 and 2 at seed 7. Explained below.


### Check 6, extra. Visit tests for bootstrap seeds 1 to 12

One alternative seed shows that the X-learner verdict can flip, but not how often. We repeat the visit tests for bootstrap seeds 1 to 12 (the same 200 paired resamples, the same Bonferroni correction over 3 learners) and count, for each learner, in how many seeds its Bonferroni p value is below 0.05. Conversion is left out to keep the run short, because no conversion test is close to the line. Each seed takes about 45 seconds.

In [9]:
SWEEP_SEEDS = list(range(1, 13))
sc_v = {mod: s5[f"{mod}_visit"].to_numpy() for mod in NAMES}
point_v = {mod: m.qini_coefficient(y5["visit"], t5, v) for mod, v in sc_v.items()}
rows = []
for seed in SWEEP_SEEDS:
    boot = m.bootstrap_metrics(y5["visit"], t5, sc_v, n_boot=N_BOOT, seed=seed)
    for mod in ("s", "t", "x"):
        r = m.paired_difference(boot, mod, "response", point_v[mod], point_v["response"])
        rows.append(dict(seed=seed, model=NAMES[mod], ci_lo=r["lo"], p_bonferroni=min(1.0, r["p_one_sided"] * 3)))
sweep = pd.DataFrame(rows)
sweep["below_005"] = sweep["p_bonferroni"] < 0.05
sweep["supported"] = (sweep["ci_lo"] > 0) & sweep["below_005"]
LEARNERS = ["S-learner", "T-learner", "X-learner"]
sweep_p = sweep.pivot(index="seed", columns="model", values="p_bonferroni")[LEARNERS]
display(sweep_p.round(4))
sweep_n = sweep.groupby("model").agg(seeds_p_below_005=("below_005", "sum"), seeds_supported=("supported", "sum"),
                                     min_p=("p_bonferroni", "min"), median_p=("p_bonferroni", "median"), max_p=("p_bonferroni", "max")).loc[LEARNERS]
display(sweep_n.round(4))
n_below = {k: int(v) for k, v in sweep_n["seeds_p_below_005"].items()}
# the seed 7 row of the sweep is the same run as the rerun above, so the two must agree
same_as_rerun = np.allclose(sweep_p.loc[7].to_numpy(), cmp6[cmp6.outcome == "visit"].p_bonferroni_seed7.to_numpy())
print("seed 7 of the sweep equals the seed 7 rerun above:", same_as_rerun)

model,S-learner,T-learner,X-learner
seed,,,
1,0.0149,0.2836,0.0448
2,0.0149,0.4179,0.0448
3,0.0149,0.3582,0.0299
4,0.0149,0.3582,0.0299
5,0.0149,0.4030,0.0448
6,0.0149,0.3284,0.0149
7,0.0149,0.2388,0.0448
8,0.0149,0.5075,0.0597
9,0.0149,0.2687,0.0448


,seeds_p_below_005,seeds_supported,min_p,median_p,max_p
model,,,,,
S-learner,12,12,0.0149,0.0149,0.0149
T-learner,0,0,0.2388,0.3582,0.5075
X-learner,10,10,0.0149,0.0448,0.0597


seed 7 of the sweep equals the seed 7 rerun above: True


### Check 6, why it fails and what it affects

In [10]:
def row6(outcome, model):
    return cmp6[(cmp6.outcome == outcome) & (cmp6.model == model)].iloc[0]
xr, sr, tr_ = row6("visit", "X-learner"), row6("visit", "S-learner"), row6("visit", "T-learner")
both = cmp6[cmp6.supported_seed42 & cmp6.supported_seed7]
neither = cmp6[~cmp6.supported_seed42 & ~cmp6.supported_seed7]
conv = cmp6[cmp6.outcome == "conversion"]
conv_min_p = float(conv[["p_bonferroni_seed42", "p_bonferroni_seed7"]].to_numpy().min())
n_seeds = len(SWEEP_SEEDS)
EXPLAIN6 = [
    f"Check 6 fails because one verdict flips. With bootstrap seed 42 (notebook 04) the X-learner on visits has a Bonferroni p value of {xr.p_bonferroni_seed42:.4f}, which is not below 0.05, so the test is not supported. "
    f"With bootstrap seed 7 it has {xr.p_bonferroni_seed7:.4f} and a lower interval end of {xr.ci_lo_seed7:.6f}, so the test is supported.",
    f"The test resolution is p = (k + 1) / ({N_BOOT} + 1), where k is the number of resamples in which the learner did not beat the response baseline. "
    f"The X-learner on visits had k = {int(xr.k_seed42)} at seed 42 and k = {int(xr.k_seed7)} at seed 7. After the Bonferroni factor of 3 this is p = {xr.p_bonferroni_seed42:.4f} and p = {xr.p_bonferroni_seed7:.4f}. "
    f"A difference of {int(xr.k_seed42 - xr.k_seed7)} resamples out of {N_BOOT} is enough to cross 0.05, so this verdict sits on the line.",
    f"The other verdicts do not move. Supported under both seeds: {', '.join(f'{r.model} on {r.outcome}' for _, r in both.iterrows()) or 'none'}. "
    f"Not supported under either seed: {', '.join(f'{r.model} on {r.outcome}' for _, r in neither.iterrows()) or 'none'}. "
    f"The S-learner on visits has Bonferroni p {sr.p_bonferroni_seed42:.4f} (seed 42) and {sr.p_bonferroni_seed7:.4f} (seed 7). "
    f"The T-learner on visits has {tr_.p_bonferroni_seed42:.4f} and {tr_.p_bonferroni_seed7:.4f}. "
    f"All six conversion p values (3 learners, 2 seeds) are at least {conv_min_p:.4f}, so none is below 0.05.",
    f"With bootstrap seeds 1 to {n_seeds} on visits, the Bonferroni p value is below 0.05 in {n_below['S-learner']} of {n_seeds} seeds for the S-learner, {n_below['T-learner']} of {n_seeds} for the T-learner and {n_below['X-learner']} of {n_seeds} for the X-learner. "
    + ("The X-learner verdict therefore depends on the bootstrap seed." if 0 < n_below["X-learner"] < n_seeds else "The X-learner verdict does not depend on the seed in this range.")
    + " Seed 7 is in this range and seed 42 is not.",
    "Consequence for the report: it must not say that the X-learner is significantly better than the response baseline on visits. It may say that the X-learner gain on visits is borderline and depends on the bootstrap seed. "
    "The S-learner gain on visits holds up under bootstrap noise. The T-learner on visits and every learner on conversions stay not supported.",
]
TABLE6 = md_table(pd.DataFrame({
    "Learner (visits)": list(sweep_n.index),
    f"Seeds with Bonferroni p below 0.05 (of {n_seeds})": [str(int(v)) for v in sweep_n["seeds_p_below_005"]],
    "Smallest p": [f"{v:.4f}" for v in sweep_n["min_p"]],
    "Median p": [f"{v:.4f}" for v in sweep_n["median_p"]],
    "Largest p": [f"{v:.4f}" for v in sweep_n["max_p"]]}))
display(Markdown("\n\n".join(EXPLAIN6) + "\n\n" + TABLE6))

Check 6 fails because one verdict flips. With bootstrap seed 42 (notebook 04) the X-learner on visits has a Bonferroni p value of 0.0746, which is not below 0.05, so the test is not supported. With bootstrap seed 7 it has 0.0448 and a lower interval end of 0.000044, so the test is supported.

The test resolution is p = (k + 1) / (200 + 1), where k is the number of resamples in which the learner did not beat the response baseline. The X-learner on visits had k = 4 at seed 42 and k = 2 at seed 7. After the Bonferroni factor of 3 this is p = 0.0746 and p = 0.0448. A difference of 2 resamples out of 200 is enough to cross 0.05, so this verdict sits on the line.

The other verdicts do not move. Supported under both seeds: S-learner on visit. Not supported under either seed: T-learner on visit, S-learner on conversion, T-learner on conversion, X-learner on conversion. The S-learner on visits has Bonferroni p 0.0149 (seed 42) and 0.0149 (seed 7). The T-learner on visits has 0.3284 and 0.2388. All six conversion p values (3 learners, 2 seeds) are at least 1.0000, so none is below 0.05.

With bootstrap seeds 1 to 12 on visits, the Bonferroni p value is below 0.05 in 12 of 12 seeds for the S-learner, 0 of 12 for the T-learner and 10 of 12 for the X-learner. The X-learner verdict therefore depends on the bootstrap seed. Seed 7 is in this range and seed 42 is not.

Consequence for the report: it must not say that the X-learner is significantly better than the response baseline on visits. It may say that the X-learner gain on visits is borderline and depends on the bootstrap seed. The S-learner gain on visits holds up under bootstrap noise. The T-learner on visits and every learner on conversions stay not supported.

| Learner (visits) | Seeds with Bonferroni p below 0.05 (of 12) | Smallest p | Median p | Largest p |
| --- | --- | --- | --- | --- |
| S-learner | 12 | 0.0149 | 0.0149 | 0.0149 |
| T-learner | 0 | 0.2388 | 0.3582 | 0.5075 |
| X-learner | 10 | 0.0149 | 0.0448 | 0.0597 |

## Check 7. Seed spread against the claimed gain

`results/seed_stability_visit.csv` holds the Qini coefficient of the response baseline and the T-learner for three seeds on visits. The spread is the maximum minus the minimum across seeds. The claimed gain is the difference to the response baseline in `results/h1_tests_5m.csv`. The check passes only when the spread of the learner is smaller than its claimed gain. The seed runs exist for the T-learner and the response baseline only, so the S and X gains cannot be tested this way. The seed runs also use 1M training rows, while the headline results use 5M.

In [11]:
seeds = pd.read_csv(RESULTS / "seed_stability_visit.csv")
spread = seeds[["response", "t_learner"]].max() - seeds[["response", "t_learner"]].min()
seeds["t_minus_response"] = seeds["t_learner"] - seeds["response"]
gain = committed[committed.outcome == "visit"].set_index("model")["diff_vs_response"]
display(seeds.round(6))
print("spread across seeds:", spread.round(6).to_dict())
print("claimed gain over the response baseline on visits:", gain.round(6).to_dict())
print(f"T-learner minus response by seed: {', '.join(f'{v:.6f}' for v in seeds['t_minus_response'])}; mean {seeds['t_minus_response'].mean():.6f}")
gain_t, spread_t = float(gain["T-learner"]), float(spread["t_learner"])
ratio_t = spread_t / gain_t
neg_t = int((seeds["t_minus_response"] < 0).sum())
record(7, "Seed spread", spread_t < gain_t,
       f"T-learner spread across {len(seeds)} seeds is {spread_t:.6f} against a claimed gain of {gain_t:.6f} ({ratio_t:.1f} times the gain), and the response spread is {spread['response']:.6f}. "
       f"The T-learner minus response is {seeds['t_minus_response'].min():.6f} to {seeds['t_minus_response'].max():.6f} across seeds. S and X have no seed runs.")

,seed,response,t_learner,t_minus_response
0,1,0.003398,0.003312,-0.000086
1,2,0.003406,0.003217,-0.000189
2,3,0.003402,0.003557,0.000155


spread across seeds: {'response': 8e-06, 't_learner': 0.00034}
claimed gain over the response baseline on visits: {'S-learner': 0.000157, 'T-learner': 0.000177, 'X-learner': 0.000285}
T-learner minus response by seed: -0.000086, -0.000189, 0.000155; mean -0.000040
Check 7, Seed spread: FAIL
  T-learner spread across 3 seeds is 0.000340 against a claimed gain of 0.000177 (1.9 times the gain), and the response spread is 0.000008. The T-learner minus response is -0.000189 to 0.000155 across seeds. S and X have no seed runs.


### Check 7, why it fails and what it affects

In [12]:
EXPLAIN7 = [
    f"Check 7 fails because the spread of the T-learner Qini coefficient across the {len(seeds)} seeds ({spread_t:.6f}) is {ratio_t:.1f} times its claimed gain over the response baseline on visits ({gain_t:.6f}). "
    f"The response baseline itself is steady, with a spread of {spread['response']:.6f}.",
    f"The sign is not stable. The T-learner minus the response baseline is {', '.join(f'{v:.6f}' for v in seeds['t_minus_response'])} for seeds {', '.join(str(s) for s in seeds['seed'])}, "
    f"negative in {neg_t} of {len(seeds)} seeds, with a mean of {seeds['t_minus_response'].mean():.6f}. Changing the training sample and the model seed can turn the T-learner from worse than the response baseline into better.",
    f"Consequence for the report: the T-learner advantage on visits must not be claimed. The bootstrap interval of the T-learner minus the response baseline already includes zero ({row6('visit', 'T-learner').ci_lo_seed42:.6f} to {row6('visit', 'T-learner').ci_hi_seed42:.6f}), and the seed runs agree.",
    f"Limits of this check. The seed runs use 1M training rows, while the headline is at 5M, so the spread at 5M may be smaller or larger. That is untested. "
    f"The S-learner and the X-learner have no seed runs, so their gains over the response baseline ({gain['S-learner']:.6f} and {gain['X-learner']:.6f}) have not been compared with a seed spread at all.",
]
display(Markdown("\n\n".join(EXPLAIN7)))

Check 7 fails because the spread of the T-learner Qini coefficient across the 3 seeds (0.000340) is 1.9 times its claimed gain over the response baseline on visits (0.000177). The response baseline itself is steady, with a spread of 0.000008.

The sign is not stable. The T-learner minus the response baseline is -0.000086, -0.000189, 0.000155 for seeds 1, 2, 3, negative in 2 of 3 seeds, with a mean of -0.000040. Changing the training sample and the model seed can turn the T-learner from worse than the response baseline into better.

Consequence for the report: the T-learner advantage on visits must not be claimed. The bootstrap interval of the T-learner minus the response baseline already includes zero (-0.000091 to 0.000457), and the seed runs agree.

Limits of this check. The seed runs use 1M training rows, while the headline is at 5M, so the spread at 5M may be smaller or larger. That is untested. The S-learner and the X-learner have no seed runs, so their gains over the response baseline (0.000157 and 0.000285) have not been compared with a seed spread at all.

## Check 8. The scores file can be reproduced

Fit the T-learner again on the first 100k rows of the training pool, with the tuned visit settings from `results/best_params.json` and the kept features, exactly as `build_scores` does. Score the holdout and compare its Qini coefficient on visits with the one computed from `t_visit` in `results/scores_100k.parquet`. They must agree to within 0.0001. The scores file stores float32, so the refit is cast to float32 as well.

In [13]:
s100 = SCORES.get(100_000)
rows_ok = s100 is not None and len(s100) == len(test_ids) and bool(np.array_equal(s100.index.to_numpy(), test_ids))
if not rows_ok:
    # a broken scores file is a FAIL row, not a crash
    record(8, "Scores file is reproducible", False,
           "scores_100k.parquet is missing or its rows are not the holdout in make_splits order, so the refit could not be compared with it.")
else:
    tr, te = take(df, pool_ids[:100_000]), take(df, test_ids)
    new = TLearner(best["visit"]).fit(tr[kept], tr[TREATMENT], tr["visit"]).predict_uplift(te[kept]).astype("float32")
    y8, t8 = te["visit"].to_numpy(), te[TREATMENT].to_numpy()
    coef_file = m.qini_coefficient(y8, t8, s100["t_visit"].to_numpy())
    coef_new = m.qini_coefficient(y8, t8, new)
    score_gap = float(np.abs(new - s100["t_visit"].to_numpy()).max())
    print(f"Qini from the file {coef_file:.6f} | refit {coef_new:.6f} | difference {abs(coef_new - coef_file):.2e} | largest score difference {score_gap:.2e}")
    record(8, "Scores file is reproducible", abs(coef_new - coef_file) <= 1e-4,
           f"T-learner on visits at 100k rows: Qini coefficient {coef_file:.6f} in the file and {coef_new:.6f} after refit, a difference of {abs(coef_new - coef_file):.2e}. The largest difference in any single score is {score_gap:.2e}. "
           f"This also shows the 100k model was trained on pool_ids[:100_000].")

Qini from the file 0.002723 | refit 0.002723 | difference 0.00e+00 | largest score difference 0.00e+00
Check 8, Scores file is reproducible: PASS
  T-learner on visits at 100k rows: Qini coefficient 0.002723 in the file and 0.002723 after refit, a difference of 0.00e+00. The largest difference in any single score is 0.00e+00. This also shows the 100k model was trained on pool_ids[:100_000].


## Check 9 (extra). The 5M scores file holds exactly the holdout users

The pipeline file for the 5M training size must have the 2,000,000 holdout row ids from `make_splits`, each once. We also confirm that the stored treatment, visit and conversion columns equal the data file at those ids.

In [14]:
ids5 = s5.index.to_numpy()
same_set = len(ids5) == HOLDOUT_SIZE and len(np.unique(ids5)) == len(ids5) and np.array_equal(np.sort(ids5), np.sort(test_ids))
same_order = np.array_equal(ids5, test_ids)
cols_match = all(np.array_equal(s5[c].to_numpy(), df[c].to_numpy()[ids5]) for c in [TREATMENT] + list(OUTCOMES))
print(f"rows {len(ids5):,} | same id set {same_set} | same order as make_splits {same_order} | treatment and outcomes match the data {cols_match}")
record(9, "5M scores file holds the holdout ids", same_set and cols_match,
       f"The 5M scores file has {len(ids5):,} rows. The id set equals the make_splits holdout: {same_set}. Same order: {same_order}. Treatment, visit and conversion equal the data file at those ids: {cols_match}.")

rows 2,000,000 | same id set True | same order as make_splits True | treatment and outcomes match the data True
Check 9, 5M scores file holds the holdout ids: PASS
  The 5M scores file has 2,000,000 rows. The id set equals the make_splits holdout: True. Same order: True. Treatment, visit and conversion equal the data file at those ids: True.


## Claims the report must not make, and `results/audit.md`

The list of claims is built from the numbers computed above. The summary table, the explanations of checks 6 and 7, the sweep table and the list are then written to `results/audit.md`.

In [15]:
sup_both = [f"{r.model} on {r.outcome}" for _, r in both.iterrows()]
conv_t = row6("conversion", "T-learner")
CLAIMS = [
    f"Do not say that the X-learner is significantly better than the response baseline on visits. Say that it is borderline and depends on the bootstrap seed: its Bonferroni p value is {xr.p_bonferroni_seed42:.4f} with seed 42 and {xr.p_bonferroni_seed7:.4f} with seed 7, and it is below 0.05 in {n_below['X-learner']} of {n_seeds} further seeds.",
    f"Do not claim that the T-learner beats the response baseline on visits. Its Bonferroni p value is {tr_.p_bonferroni_seed42:.4f} at seed 42 and it is below 0.05 in {n_below['T-learner']} of {n_seeds} further seeds. Its seed spread of {spread_t:.6f} is {ratio_t:.1f} times its claimed gain of {gain_t:.6f}, and its difference to the response baseline is negative in {neg_t} of {len(seeds)} seed runs.",
    f"Do not claim that any uplift learner beats the response baseline on conversions. All 3 conversion tests are not supported at both bootstrap seeds. The T-learner on conversions is below the response baseline ({conv_t.diff_vs_response_seed42:.6f}, interval {conv_t.ci_lo_seed42:.6f} to {conv_t.ci_hi_seed42:.6f}).",
    f"Do not say that H1 is supported in general. The tests supported at both bootstrap seeds are: {', '.join(sup_both) or 'none'}. Nothing else may be reported as supported.",
    f"Do not describe the S-learner gain on visits as stable across training samples. It holds under bootstrap noise (p below 0.05 in {n_below['S-learner']} of {n_seeds} further seeds), but the S-learner and the X-learner have no seed runs, so the effect of refitting on another training sample is untested for them.",
    "Do not say that the seed spread measured at 1M training rows also holds at 5M, and do not say that it does not. The seed runs are at 1M rows and the headline is at 5M, so the 5M spread is untested.",
]
display(Markdown("\n\n".join(f"{i}. {c}" for i, c in enumerate(CLAIMS, 1))))

1. Do not say that the X-learner is significantly better than the response baseline on visits. Say that it is borderline and depends on the bootstrap seed: its Bonferroni p value is 0.0746 with seed 42 and 0.0448 with seed 7, and it is below 0.05 in 10 of 12 further seeds.

2. Do not claim that the T-learner beats the response baseline on visits. Its Bonferroni p value is 0.3284 at seed 42 and it is below 0.05 in 0 of 12 further seeds. Its seed spread of 0.000340 is 1.9 times its claimed gain of 0.000177, and its difference to the response baseline is negative in 2 of 3 seed runs.

3. Do not claim that any uplift learner beats the response baseline on conversions. All 3 conversion tests are not supported at both bootstrap seeds. The T-learner on conversions is below the response baseline (-0.000132, interval -0.000212 to -0.000055).

4. Do not say that H1 is supported in general. The tests supported at both bootstrap seeds are: S-learner on visit. Nothing else may be reported as supported.

5. Do not describe the S-learner gain on visits as stable across training samples. It holds under bootstrap noise (p below 0.05 in 12 of 12 further seeds), but the S-learner and the X-learner have no seed runs, so the effect of refitting on another training sample is untested for them.

6. Do not say that the seed spread measured at 1M training rows also holds at 5M, and do not say that it does not. The seed runs are at 1M rows and the headline is at 5M, so the 5M spread is untested.

In [16]:
audit = pd.DataFrame(AUDIT)
display(audit)
n_pass = int((audit["result"] == "PASS").sum())
lines = ["# Audit of the results", "",
         "Written by `notebooks/12_audit.ipynb`. The file is regenerated every time the notebook runs, so do not edit it by hand.", "",
         "| Check | Result | Comment |", "| --- | --- | --- |"]
lines += [f"| {r.check} | {r.result} | {r.comment} |" for r in audit.itertuples()]
lines += ["", f"{n_pass} of {len(audit)} checks pass."]
failed = audit[audit["result"] == "FAIL"]
if len(failed):
    lines += ["", "Failed checks: " + ", ".join(failed["check"].str.split(".").str[0]) + ". A claim that depends on a failed check must be dropped or qualified in the report."]
lines += ["", "## Check 6 explained: bootstrap stability", ""]
for p in EXPLAIN6:
    lines += [p, ""]
lines += [f"Visit tests for bootstrap seeds 1 to {n_seeds}, how often the Bonferroni p value is below 0.05 (same 200 resamples, same correction over 3 learners):", "", TABLE6, ""]
lines += ["## Check 7 explained: seed spread", ""]
for p in EXPLAIN7:
    lines += [p, ""]
lines += ["## Claims the report must not make", ""]
lines += [f"{i}. {c}" for i, c in enumerate(CLAIMS, 1)]
(RESULTS / "audit.md").write_text("\n".join(lines) + "\n")
print(f"{n_pass} of {len(audit)} checks pass; wrote results/audit.md")

,check,result,comment
0,1. No leakage,PASS,"The holdout has 2,000,000 unique ids and share..."
1,2. Treated share,PASS,Treated share runs from 0.8497 to 0.8506 acros...
2,3. Curve end equals the average effect,PASS,The curve end matches the observed holdout ave...
3,4. Intervals contain the estimates,PASS,8 of 8 rows have ci_lo <= qini_coef <= ci_hi. ...
4,5. Bounded by the oracle,PASS,Committed planted test: the oracle gives 0.006...
5,6. Bootstrap stability,FAIL,Seed 7 against seed 42 on 6 learner and outcom...
6,7. Seed spread,FAIL,T-learner spread across 3 seeds is 0.000340 ag...
7,8. Scores file is reproducible,PASS,T-learner on visits at 100k rows: Qini coeffic...
8,9. 5M scores file holds the holdout ids,PASS,"The 5M scores file has 2,000,000 rows. The id ..."


7 of 9 checks pass; wrote results/audit.md
